<a href="https://colab.research.google.com/github/MukulVerma-ML/Titanic-Survival-Predictor/blob/main/Day_(8%2C9)_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , classification_report , confusion_matrix

In [34]:
df=pd.read_csv("IMDB_Dataset[1].csv")

print(df.shape)
print(df.head())

(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [35]:
df['review'].value_counts()
df.isnull().sum()

,0
review,0
sentiment,0


In [36]:
def clean_text(text):

  text = re.sub(r'<.*!?>',' ',text)
  text = re.sub(r'[^a-zA-Z]',' ',text)

  text=text.lower()

  text=' '.join(text.split())

  return text

In [37]:
df['clean_review']= df['review'].apply(clean_text)
print("Befor_cleaning:", df['review'][0][:100])
print("After_cleaning:", df['clean_review'][0][:100])



Befor_cleaning: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. The
After_cleaning: one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they a


In [38]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})
print(df.head())
# print(df['sentiment'].head())
# print(df['sentiment'].unique())

                                              review  ...                                       clean_review
0  One of the other reviewers has mentioned that ...  ...  one of the other reviewers has mentioned that ...
1  A wonderful little production. <br /><br />The...  ...  a wonderful little production the realism real...
2  I thought this was a wonderful way to spend ti...  ...  i thought this was a wonderful way to spend ti...
3  Basically there's a family where a little boy ...  ...  basically there s a family where a little boy ...
4  Petter Mattei's "Love in the Time of Money" is...  ...  petter mattei s love in the time of money is a...

[5 rows x 3 columns]


In [40]:
#training model
X=df['clean_review']
Y=df['sentiment']

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)
print("X_train shape:",X_train.shape)
print("X_test shape:",X_test.shape)
print("Y_train shape:",Y_train.shape)
print("Y_test shape:",Y_test.shape)

X_train shape: (40000,)
X_test shape: (10000,)
Y_train shape: (40000,)
Y_test shape: (10000,)


In [43]:
# text to numbers
tfidf=TfidfVectorizer(max_features=5000)

#transorm data in tfidf

X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf)

(40000, 5000)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 777853 stored elements and shape (10000, 5000)>
  Coords	Values
  (0, 85)	0.14321890511278773
  (0, 143)	0.05675299122341053
  (0, 166)	0.11930083858952995
  (0, 191)	0.05289369475997725
  (0, 204)	0.10377870751947162
  (0, 380)	0.04191526510768298
  (0, 412)	0.04567738831198435
  (0, 422)	0.055694910566225844
  (0, 471)	0.10003775238212977
  (0, 472)	0.07803625524149915
  (0, 490)	0.06410871052721534
  (0, 548)	0.08528696377528877
  (0, 590)	0.024003187876137103
  (0, 637)	0.10959247447813754
  (0, 711)	0.08677360076569154
  (0, 970)	0.04378610761924023
  (0, 1159)	0.07842652697751919
  (0, 1170)	0.101531061390609
  (0, 1195)	0.05220303124864251
  (0, 1332)	0.07628155966256192
  (0, 1422)	0.061372392615945605
  (0, 1487)	0.27471945354480454
  (0, 1490)	0.04503134412318799
  (0, 1491)	0.05366785277278811
  (0, 1613)	0.09312573307781626
  :	:
  (9999, 3200)	0.06558477760589895
  (9999, 3538)	0.18018965965891304


In [45]:
model=LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf,Y_train)
print("Model is trained")

Model is trained


In [46]:
X_pred=model.predict(X_test_tfidf)

print("Accuracy:",accuracy_score(Y_test,X_pred))
print("Detailed_Report:")
print(classification_report(Y_test,X_pred))
print("Confusion_Matrix:")
print(confusion_matrix(Y_test,X_pred))

Accuracy: 0.8544
Detailed_Report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      4961
           1       0.85      0.87      0.86      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

Confusion_Matrix:
[[4174  787]
 [ 669 4370]]


In [47]:
def predict_sentiment(review):
  cleaned=clean_text(review)
  vector=tfidf.transform([cleaned])
  prediction= model.predict(vector)[0]

  return "positive" if prediction==1 else "negative"

In [48]:
test1="This movie was absolutely fantastic."
test2="Worst movie ever."

print("Custom_Test:")
print(f"Review:{test1} -> {predict_sentiment(test1)}")
print(f"Review:{test2} -> {predict_sentiment(test2)}")


Custom_Test:
Review:This movie was absolutely fantastic. -> positive
Review:Worst movie ever. -> negative
